In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [120]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction import DictVectorizer

In [4]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'

In [5]:
# !wget $data

In [6]:
df = pd.read_csv('course_lead_scoring_2026.csv')

In [7]:
df

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1
...,...,...,...,...,...,...,...,...,...
4995,organic_search,education,employed,asia,45834.0,2,2,0.37,1
4996,NaN,technology,employed,north_america,84831.0,2,5,0.51,0
4997,organic_search,finance,employed,asia,53636.0,4,6,0.75,1
4998,referral,technology,employed,africa,69430.0,4,8,0.79,1


In [9]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [12]:
categorical = df.columns[df.dtypes == 'str']
categorical = list(categorical)
categorical

['lead_source', 'industry', 'employment_status', 'location']

In [144]:
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score'] 

In [17]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [18]:
df.nunique()

lead_source                    5
industry                       6
employment_status              4
location                       5
annual_income               4313
number_of_courses_viewed       7
interaction_count             14
lead_score                    99
converted                      2
dtype: int64

In [19]:
df[categorical] = df[categorical].fillna('NA')

In [21]:
df[categorical].isnull().sum()

lead_source          0
industry             0
employment_status    0
location             0
dtype: int64

In [23]:
df[numerical] = df[numerical].fillna(0.0)

In [24]:
df[numerical].isnull().sum()

annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
dtype: int64

In [25]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [27]:
df.industry.unique()

<StringArray>
[   'technology',        'retail', 'manufacturing',     'education',
    'healthcare',            'NA',       'finance']
Length: 7, dtype: str

In [30]:
df.industry.value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [28]:
df.industry.mode()

0    technology
Name: industry, dtype: str

In [40]:
for i in range(len(numerical)):
    print(numerical[i])
    print("-------------------------")
    print(df[numerical].corrwith(df[numerical[i]]))
    print()

annual_income
-------------------------
annual_income               1.000000
number_of_courses_viewed    0.161300
interaction_count           0.122842
lead_score                  0.229496
dtype: float64

number_of_courses_viewed
-------------------------
annual_income               0.161300
number_of_courses_viewed    1.000000
interaction_count           0.721609
lead_score                  0.757204
dtype: float64

interaction_count
-------------------------
annual_income               0.122842
number_of_courses_viewed    0.721609
interaction_count           1.000000
lead_score                  0.915746
dtype: float64

lead_score
-------------------------
annual_income               0.229496
number_of_courses_viewed    0.757204
interaction_count           0.915746
lead_score                  1.000000
dtype: float64



In [42]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [44]:
len(df_train), len(df_val), len(df_test), len(df)

(3000, 1000, 1000, 5000)

In [46]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

In [53]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [92]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [95]:
df_test

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score
0,social_media,retail,employed,south_america,41247.0,3,7,0.61
1,organic_search,healthcare,employed,north_america,70825.0,3,6,0.62
2,social_media,healthcare,employed,south_america,58488.0,3,4,0.56
3,NA,education,NA,europe,30138.0,1,4,0.48
4,NA,retail,employed,asia,53869.0,2,3,0.49
...,...,...,...,...,...,...,...,...
995,organic_search,manufacturing,employed,NA,65099.0,1,4,0.53
996,social_media,healthcare,student,europe,38954.0,0,0,0.15
997,organic_search,education,self_employed,asia,48768.0,2,5,0.47
998,social_media,NA,student,asia,0.0,2,2,0.35


In [60]:
categorical

['lead_source', 'industry', 'employment_status', 'location']

In [96]:
def mutual_info_converted_score(series):
    return mutual_info_score(y_train, series)

In [97]:
mi = df_train[categorical].apply(mutual_info_converted_score)
mi.sort_values(ascending=False)
round(mi, 2)

lead_source          0.03
industry             0.00
employment_status    0.02
location             0.00
dtype: float64

In [86]:
dv = DictVectorizer(sparse=False)

In [102]:
dict_train = df_train[categorical+numerical].to_dict(orient='records')

In [103]:
dict_train

[{'lead_source': 'paid_ads',
  'industry': 'finance',
  'employment_status': 'employed',
  'location': 'asia',
  'annual_income': 65980.0,
  'number_of_courses_viewed': 2,
  'interaction_count': 3,
  'lead_score': 0.37},
 {'lead_source': 'events',
  'industry': 'finance',
  'employment_status': 'employed',
  'location': 'south_america',
  'annual_income': 71662.0,
  'number_of_courses_viewed': 0,
  'interaction_count': 0,
  'lead_score': 0.09},
 {'lead_source': 'referral',
  'industry': 'NA',
  'employment_status': 'employed',
  'location': 'asia',
  'annual_income': 40220.0,
  'number_of_courses_viewed': 2,
  'interaction_count': 5,
  'lead_score': 0.55},
 {'lead_source': 'organic_search',
  'industry': 'finance',
  'employment_status': 'employed',
  'location': 'north_america',
  'annual_income': 88352.0,
  'number_of_courses_viewed': 2,
  'interaction_count': 2,
  'lead_score': 0.45},
 {'lead_source': 'events',
  'industry': 'technology',
  'employment_status': 'employed',
  'locati

In [104]:
dv.fit(dict_train)

DictVectorizer(sparse=False)

In [105]:
X_train = dv.transform(dict_train)

In [106]:
X_train

array([[6.5980e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        2.0000e+00],
       [7.1662e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 1.0000e+00,
        0.0000e+00],
       [4.0220e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        2.0000e+00],
       ...,
       [7.1532e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        3.0000e+00],
       [4.8836e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00],
       [6.8969e+04, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00]], shape=(3000, 28))

In [108]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

In [109]:
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l

In [113]:
dict_val = df_val[categorical+numerical].to_dict(orient='records')
X_val = dv.transform(dict_val)
y_pred = model.predict_proba(X_val)[:,1]

In [115]:
convert_decision = (y_pred > 0.5).astype(int)
convert_decision[:10]

array([1, 0, 1, 1, 1, 1, 1, 1, 1, 1])

In [117]:
y_val[:10]

array([1, 1, 0, 1, 0, 1, 1, 0, 1, 1])

In [124]:
acc_score = accuracy_score(convert_decision, y_val)
round(acc_score, 2)

0.65

In [126]:
acc_score = accuracy_score(convert_decision, y_val)
acc_score

0.645

In [128]:
categorical1 = ['industry', 'employment_status', 'location']

In [129]:
dict_train = df_train[categorical1+numerical].to_dict(orient='records')
dv.fit(dict_train)
X_train = dv.transform(dict_train)
model.fit(X_train, y_train)

dict_val = df_val[categorical1+numerical].to_dict(orient='records')
X_val = dv.transform(dict_val)
y_pred = model.predict_proba(X_val)[:,1]
convert_decision = (y_pred > 0.5).astype(int)
acc_score1 = accuracy_score(convert_decision, y_val)
acc_score1

0.642

In [148]:
a = acc_score - acc_score1

In [133]:
numerical1 = ['annual_income',
 'interaction_count',
 'lead_score']

In [137]:
numerical1

['annual_income', 'interaction_count', 'lead_score']

In [138]:
dict_train = df_train[categorical+numerical1].to_dict(orient='records')
dv.fit(dict_train)
X_train = dv.transform(dict_train)
model.fit(X_train, y_train)

dict_val = df_val[categorical+numerical1].to_dict(orient='records')
X_val = dv.transform(dict_val)
y_pred = model.predict_proba(X_val)[:,1]
convert_decision = (y_pred > 0.5).astype(int)
acc_score2 = accuracy_score(convert_decision, y_val)
acc_score2

0.643

In [149]:
b = acc_score - acc_score2

In [142]:
numerical2 = ['annual_income',
 'number_of_courses_viewed',
 'lead_score']

In [146]:
dict_train = df_train[categorical+numerical2].to_dict(orient='records')
dv.fit(dict_train)
X_train = dv.transform(dict_train)
model.fit(X_train, y_train)

dict_val = df_val[categorical+numerical2].to_dict(orient='records')
X_val = dv.transform(dict_val)
y_pred = model.predict_proba(X_val)[:,1]
convert_decision = (y_pred > 0.5).astype(int)
acc_score3 = accuracy_score(convert_decision, y_val)
acc_score3

0.601

In [150]:
c = acc_score - acc_score3

In [155]:
a, b, c

(0.0030000000000000027, 0.0020000000000000018, 0.04400000000000004)

In [154]:
np.array([a, b, c]).min()

np.float64(0.0020000000000000018)

In [158]:
dict_train = df_train[categorical+numerical].to_dict(orient='records')
dv.fit(dict_train)
X_train = dv.transform(dict_train)
dict_val = df_val[categorical+numerical].to_dict(orient='records')
X_val = dv.transform(dict_val)

for C in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict_proba(X_val)[:,1]
    convert_decision = (y_pred > 0.5).astype(int)
    score = accuracy_score(convert_decision, y_val)
    print(C, score)

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
